# 🧪 算子优化实验（Operator_Optimization_Ascend）

**从已有实验中找一个算子，完成"优化前 → 优化后"的完整对比**

## 学习目标

1. 理解算子（Operator）在深度学习框架中的作用
2. 掌握算子优化的一般流程：定位算子 → 分析瓶颈 → 实施优化 → 前后对比
3. 从 MobileNetV3 中选择 Conv+BN 算子对，完成 BN 折叠（算子融合）优化
4. 在 Ascend NPU 或 CPU 上运行优化前后推理，对比时延、吞吐、显存与算子数量
5. 理解昇腾 CANN / ATC 图编译阶段算子融合的底层思路

## 实验设计

本实验只优化一个算子：MobileNetV3 中的 `Conv2d + BatchNorm2d` 算子对。

- 优化对象：推理阶段的 BN 折叠，把 BN 的 scale/shift 折进 Conv 权重
- 优化方法：算子融合
- 对比方式：同一模型、同一输入，分别运行"融合前"与"融合后"推理
- 评估指标：正确性、平均时延、吞吐、峰值显存、算子数量

## 为什么需要算子优化

深度学习模型最终会被编译成一系列算子（如 Conv、MatMul、Softmax 等）在硬件上执行。算子的实现方式直接影响：

- **执行时延**：算子内核是否高效、是否被频繁启动
- **显存占用**：中间张量是否会反复落地到 HBM
- **硬件利用率**：AI Core / Cube Unit 是否被充分利用

常见优化手段：

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">方法</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">说明</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">例子</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">算子融合</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">把多个连续算子合成一个内核，减少启动与数据搬运</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">Conv+BN 融合</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">Tiling / 分块</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">把大张量切块，提升缓存命中</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">大矩阵分块计算</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">低精度</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">用 FP16 / BF16 代替 FP32</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">昇腾混合精度推理</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">内存复用</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">减少中间张量的峰值显存</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">算子内 workspace 复用</td>
    </tr>
  </tbody>
</table>

## 为什么选择 Conv+BN

MobileNetV3 的每个 MobileBlock 基本都包含 1×1 Conv+BN、Depthwise Conv+BN 和 1×1 Conv+BN，整个模型中有大量可融合算子对。BN 折叠是数学等价变换：

$$W' = \frac{\gamma \cdot W}{\sqrt{\text{running\_var} + \epsilon}}, \quad b' = \frac{\gamma \cdot (b - \text{running\_mean})}{\sqrt{\text{running\_var} + \epsilon}} + \beta$$

融合后 BN 算子被完全移除，输出与原始模型一致，非常适合作为算子优化的入门案例。

> 💡 如果其他实验或华为 ModelZoo（https://www.hiascend.com/software/modelzoo）中的模型也有适合优化的算子，
> 可以按同样的"优化前跑一次、优化后跑一次"方式复现本实验方法。

## 实验流程

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">步骤</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">Notebook</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">内容</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">0</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">00</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">实验总览与环境准备</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">1</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">01</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">定位 MobileNetV3 中的 Conv+BN 算子对</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">2</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">02</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">未融合模型推理（基线）</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">3</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">03</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">Conv+BN 融合并验证</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">4</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">04</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">优化前后对比与结论</td>
    </tr>
  </tbody>
</table>

In [ ]:
import torch

try:
    import torch_npu  # noqa: F401
    npu_available = torch.npu.is_available()
except Exception:
    npu_available = False

print(f"PyTorch: {torch.__version__}")
print(f"NPU available: {npu_available}")
if npu_available:
    print(f"NPU: {torch.npu.get_device_name(0)}")

try:
    import matplotlib
    print(f"Matplotlib: {matplotlib.__version__}")
except Exception as exc:
    print(f"Matplotlib 不可用: {exc}")

## 课后练习

1. (单选题) 算术强度（FLOPs/Bytes）低于硬件 ridge point 的算子属于？
   - A. memory-bound
   - B. compute-bound
   - C. latency-bound
   - D. 无法判断

2. (单选题) 小 batch 推理中，Conv+BN 融合收益主要来自？
   - A. 减少 kernel 启动与中间读写
   - B. 减少 FLOPs
   - C. 提高精度
   - D. 减少参数

3. (多选题) 合法且常见的算子优化手段包括？
   - A. 算子融合
   - B. 低精度
   - C. Tiling 分块
   - D. 增加中间张量

4. (多选题) 判断优化方案是否有效需要结合？
   - A. 硬件峰值算力与带宽
   - B. 算术强度
   - C. batch_size 与 kernel 启动开销
   - D. 数据加载方式

5. (判断题) 算术强度低于 ridge point 说明算子是 compute-bound。

6. (判断题) 昇腾 CANN/ATC 在图编译阶段可能自动执行算子融合。

7. (填空题) 硬件 ridge point 的计算公式为 ____。

8. (填空题) 对 memory-bound 算子，优化重点是减少 ____；对 compute-bound 算子，重点是提升 ____。

9. (简答题) 为什么基准测试需要 warmup、repeats 与同步？

10. (简答题) 给定一个算子的算术强度，如何判断该用融合还是低精度？

11. (代码设计题) 编写 conv2d_benchmark 函数，返回平均耗时与吞吐。

12. (单选题) 某算子算术强度 10 FLOPs/B，硬件 ridge point 20 FLOPs/B，它属于？
   - A. memory-bound
   - B. compute-bound
   - C. balanced
   - D. 无法确定

13. (多选题) 算子融合的潜在收益包括？
   - A. 减少 kernel 启动
   - B. 减少中间张量读写
   - C. 降低峰值显存
   - D. 保证精度提升

14. (判断题) 算子融合在任何 batch_size 下都会提升吞吐。

15. (简答题) 设计一个 NPU 上的融合前后对比实验，写出变量控制与结论判断标准。

> 参考答案见 answer/06.02_experiment_overview_answer.ipynb。